This is a classification model that is trained to predict if an image has good quality for the lipids to even be segmented.

In [ ]:
from fastai.vision.all import *
from utils import (
    generic_image_path,
    generic_image_path_processed,
    generic_segmentation_path,
    generic_segmentation_path_processed,
    paste_imgs,
)
np.int = np.int32 # Need to add this to make preset model work

In [ ]:
# In order to not keep changing the folder names in the utils file. I organized the "data/" folder to be inside different resolution scenario folders.
# So revise the root path assigned when necessary.
root = Path("../uvp6_lipid_sac_classification")

# df = pd.read_csv("metadata.csv")
df = pd.read_csv("../uvp6_lipid_sac_classification/metadata_uvp6_lipid.csv")

image_path_processed = generic_image_path_processed(root)

def label_func(p:Path):
    return segmentation_path_processed(p.name)

df['image_file_processed'] = df.filename.apply(image_path_processed)

# print(df[1:5])

### Training

In [ ]:
# Create a new data block that is a categoryblock instead of a maskblock
dls =  DataBlock(
    blocks=(ImageBlock,CategoryBlock),
    get_x = ColReader("image_file_processed"),
    get_y = ColReader("label"),

    # need to convert all images to the same size
    item_tfms=Resize(224, method=ResizeMethod.Pad, pad_mode='zeros'), # aparently 224x224 is optimal for resnet34

    # TODO Add a batch norm?
    batch_tfms=[Normalize.from_stats(*imagenet_stats), *aug_transforms(pad_mode='zeros', max_rotate=180)],
    
    splitter=RandomSplitter(valid_pct=0.2, seed=88)
    # splitter=ColSplitter("is_valid"),
).dataloaders(df, batch_size=8, drop_last=True)

In [ ]:
# Create a file that identified which images were used for training or testing
image_files = df["image_file_processed"].tolist()

# Use the same RandomSplitter as in the DataBlock
splitter = RandomSplitter(valid_pct=0.2, seed=88)
train_idx, valid_idx = splitter(image_files)

# Create a DataFrame to store split information
split_data = pd.DataFrame({
    "image_file": image_files,
    "split": ["train" if i in train_idx else "test" for i in range(len(image_files))]
})

# Save to CSV
split_data.to_csv(os.path.join(root,"dataset_split.csv"), index=False)

print("Dataset split saved to dataset_split.csv")

In [ ]:
# Create learner with default loss functions and metrics
learn = cnn_learner(dls, resnet34, metrics=accuracy )

cbs = []
cbs.append(EarlyStoppingCallback(patience=5))
cbs.append(SaveModelCallback(fname="model_resnet34_lipidcateg")) 
cbs.append(GradientAccumulation(n_acc=8))


learn.fine_tune(40, cbs = cbs)

# # To identify the optimal learning rate if necessary:
# # learn.lr_find()


In [ ]:
learn.summary()
# For exploring the model res") # Should this ults
learn.show_results(figsize =(10,10))

In [ ]:
# Save load validate
learn.export("models/resnet34_lipid_classifier.pkl")
learn.save("resnet34_lipid_classifier")  # Saves in 'models/resnet34_classifier.pth'

learn = cnn_learner(dls, resnet34, metrics=accuracy)  # Recreate the learner
learn.load("resnet34_lipid_classifier")  # Load saved model

valid_loss, valid_acc = learn.validate()
print(f"Validation Loss: {valid_loss:.4f}, Validation Accuracy: {valid_acc:.4f}")


In [ ]:
# Test model with new data
img = PILImage.create("../downscaled_4/data/images_processed/20130815 104223 853 000000 1067 0284.bmp")  # Load an image
display(img)
pred, pred_idx, probs = learn.predict(img)

print(f"Predicted Class: {pred}")
print(f"Confidence Scores: {probs}")

In [ ]:
# Output the predictions

# Get predictions on the validation dataset
dl = learn.dls.valid
preds, targs, *_ = learn.get_preds(dl=dl, with_decoded=True)

# Map predictions and targets to class names
predicted_classes = [learn.dls.vocab[i] for i in preds.argmax(dim=1)]
actual_classes = [learn.dls.vocab[i] for i in targs]

# Get filenames correctly
if isinstance(dl.items, pd.DataFrame):
    items = dl.items["filename"].tolist()  # Extract from DataFrame column "filename"
else:
    items = list(dl.items) if hasattr(dl, "items") else list(dl.dataset.items)

# Ensure lengths match
if len(items) == len(actual_classes) == len(predicted_classes):
    df = pd.DataFrame({
        'Filename': items,
        'Annotated Class': actual_classes,
        'Predicted Class': predicted_classes,
        'Data Split': 'validation'
    })

    # Save to CSV)
    df.to_csv(os.path.join(root,"model_results_test.csv"), index=False)
    print("CSV file 'model_results_test.csv' has been created successfully.")
else:
    raise ValueError(f"Data length mismatch! Filenames({len(items)}), Actual({len(actual_classes)}), Predicted({len(predicted_classes)})")

# !!! For some reason there is an error for the training data...
# Get predictions on the training dataset
# dl = learn.dls.train
# preds, targs, *_ = learn.get_preds(dl=dl, with_decoded=True)

# # Map predictions and targets to class names
# predicted_classes = [learn.dls.vocab[i] for i in preds.argmax(dim=1)]
# actual_classes = [learn.dls.vocab[i] for i in targs]

# # Get filenames correctly
# if isinstance(dl.items, pd.DataFrame):
#     items = dl.items["filename"].tolist()  # Extract from DataFrame column "filename"
# else:
#     items = list(dl.items) if hasattr(dl, "items") else list(dl.dataset.items)

# # Ensure lengths match
# if len(items) == len(actual_classes) == len(predicted_classes):
#     df = pd.DataFrame({
#         'Filename': items,
#         'Annotated Class': actual_classes,
#         'Predicted Class': predicted_classes,
#         'Data Split': 'train'
#     })

#     # Save to CSV)
#     df.to_csv(os.path.join(root,"model_results_train.csv"), index=False)
#     print("CSV file 'model_results_train.csv' has been created successfully.")
# else:
#     raise ValueError(f"Data length mismatch! Filenames({len(items)}), Actual({len(actual_classes)}), Predicted({len(predicted_classes)})")


In [ ]:
# *** Inference on new images ***
# 1. Downscaled LOKI2013: python_workspace\downscaled_4\data\images_processed
# 2. UVP6 Amundsen 2023 part 2

model = load_learner("models/resnet34_lipid_classifier.pkl", cpu=True)  # File models/learner.pkl
model.load("resnet34_lipid_classifier")  # File models/model_resnet34.pth
model.dls.device = 'cpu' # PP? Why is it ran with the cpu and not the gpu?

In [ ]:
***** OLD SCRIPTS *****